# W4D3 — Detection with YOLO — Lab

**Week 4 · Day 3 · CNNs & Model Fine-Tuning** · Lab

Every model you have built so far answers a question of fixed shape: ten probabilities, or one
number. Today's model answers *"what is in this picture, and where"* — and it decides how long the
answer is. One photograph returns three boxes, the next returns forty-one.

Running the detector takes one line. The lab is what comes after it:

- **IoU**, written by hand, reproducing the morning's `0.143`, `0.39`, `0.60`.
- **A threshold sweep**, because "the model found 180 objects" is not a result until you say at what
  confidence, and what it cost you in precision.
- **NMS turned off**, on a photograph where the detector puts 33 boxes on 4 cars.
- **mAP@0.5 against ground truth**, which is the number a client actually asks for.
- **Two photographs it fails on**, which have been in the dataset since Monday waiting for today.

<div dir="rtl" align="right">

# الأسبوع ٤ · اليوم ٣ — الكشف بـYOLO

**الأسبوع الرابع · اليوم الثالث · الشبكات الالتفافية وضبط النماذج** · معمل

كل نموذج بنيته حتى الآن يجيب عن سؤال ثابت الشكل: عشرة احتمالات أو عدد واحد. أما نموذج اليوم فيجيب عن
*«ماذا في هذه الصورة وأين»* — وهو الذي يقرّر طول الإجابة. فصورة تُرجع ثلاثة صناديق وأخرى تُرجع أحدًا
وأربعين.

وتشغيل الكاشف سطر واحد. أما المعمل فهو ما يأتي بعده:

- **تداخل الصناديق (IoU)** مكتوبًا بيدك، يُعيد إنتاج أرقام الصباح `٠٫١٤٣` و`٠٫٣٩` و`٠٫٦٠`.
- **مسح العتبات**، لأن «وجد النموذج ١٨٠ كائنًا» ليست نتيجة حتى تقول عند أي ثقة، وبكم من الدقة دفعت.
- **إطفاء كبت غير الأقصى** على صورة يضع فيها الكاشف ٣٣ صندوقًا على ٤ سيارات.
- **mAP@0.5 مقابل المرجع**، وهو الرقم الذي يسأل عنه العميل فعلًا.
- **صورتان يفشل فيهما**، وهما في البيانات منذ الاثنين تنتظران اليوم.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Read a detector's raw output — logits, normalised centre-form boxes, a fixed number of queries —
  and convert it to pixel `(x1, y1, x2, y2)` yourself.
- Write `iou(box_a, box_b)` and get the lecture's three numbers.
- Sweep a confidence threshold and say what precision and recall each setting bought.
- Explain what non-max suppression removes, having counted the boxes with and without it.
- Compute mAP@0.5 and mAP@0.75 against ground truth, and say why the second is lower.
- Look at two failures and name the failure mode instead of the metric.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تقرأ الخرج الخام لكاشف — لوغيتات، وصناديق مُطبَّعة بصيغة المركز، وعدد استعلامات ثابت — وأن
  تحوّله بنفسك إلى `(x1, y1, x2, y2)` بالبكسل.
- أن تكتب `iou(box_a, box_b)` فتحصل على أرقام المحاضرة الثلاثة.
- أن تمسح عتبة الثقة وتقول ماذا اشترت كل قيمة من الدقة والاستدعاء.
- أن تشرح ما الذي يحذفه كبت غير الأقصى، بعد أن عددت الصناديق بوجوده وبغيابه.
- أن تحسب mAP@0.5 وmAP@0.75 مقابل المرجع، وتقول لماذا الثاني أقلّ.
- أن تنظر إلى فشلين فتسمّي نمط الفشل بدل أن تسمّي المقياس.

</div>

## About the data

**Datasets:** `sample_photos` — the same twenty photographs you convolved on Monday — and
`detection_boxes`, their ground truth.

`boxes.csv` holds 554 boxes over 120 images in three classes: **person** (196), **car** (316) and
**stop sign** (42). Sixty-seven of those boxes are on the twenty photos you already know; the other
hundred images ship inside `detection_boxes.zip` for the stretch section.

One row is one object: an image filename, a class, and four numbers. **The four numbers are pixels
in `(x1, y1, x2, y2)` at the stored 640 px resolution.** `yolos-tiny` returns normalised centre-form
boxes — `(cx, cy, w, h)`, each between 0 and 1 — so every comparison in this notebook goes through a
conversion. Skip it and your IoUs come out near zero while looking like plausible small numbers.

**Two known problems, both deliberate.**

1. **The ground truth drops boxes smaller than 32×32.** A detector cannot find a person who is nine
   pixels tall, and scoring against boxes no model can hit measures the annotator, not the model.
   The consequence is a rule you must follow in task 2.2: filter your *predictions* by the same
   minimum size, or the small boxes the detector does emit are all counted as false positives.
   Precision at confidence 0.5 goes from 0.19 to 0.46 on that one line.
2. **Two of the twenty photographs are hard on purpose**, and `HARD_IMAGES.txt` names them. They
   were chosen by measurement: `yolos-tiny` finds **0%** of the labelled objects in one and **33%**
   in the other, at confidence 0.5. Task 2.5 is about those two.

**First run downloads** `hustvl/yolos-tiny` from Hugging Face — about 26 MB, cached afterwards.
Inference on all twenty photographs takes about a second on a laptop CPU.

<div dir="rtl" align="right">

## عن البيانات

**مجموعتا البيانات:** `sample_photos` — الصور العشرون نفسها التي التففتها يوم الاثنين — و
`detection_boxes` وهي مرجعها.

يحمل `boxes.csv` ٥٥٤ صندوقًا على ١٢٠ صورة في ثلاث فئات: **شخص** (١٩٦) و**سيارة** (٣١٦) و**لوحة قف**
(٤٢). وسبعة وستون من هذه الصناديق على الصور العشرين التي تعرفها، والمئة صورة الأخرى داخل
`detection_boxes.zip` للقسم الإضافي.

الصف الواحد كائن واحد: اسم ملف الصورة، وفئة، وأربعة أعداد. **والأعداد الأربعة بالبكسل بصيغة
`(x1, y1, x2, y2)`** عند دقة ٦٤٠ بكسل المخزَّنة. أما `yolos-tiny` فيُرجع صناديق مُطبَّعة بصيغة المركز
`(cx, cy, w, h)` كلٌّ بين صفر وواحد — فكل مقارنة في هذا الدفتر تمرّ بتحويل. وإن أهملته خرجت قيم
التداخل قرب الصفر وهي تبدو أعدادًا صغيرة معقولة.

**مشكلتان معروفتان، وكلتاهما مقصودة.**

١. **يُسقِط المرجع الصناديق الأصغر من ٣٢×٣٢.** فالكاشف لا يجد شخصًا ارتفاعه تسعة بكسلات، والتقييم
   مقابل صناديق لا يبلغها أي نموذج يقيس المُعلِّم لا النموذج. والنتيجة قاعدة تلتزمها في المهمة ٢٫٢:
   رشِّح **تنبّؤاتك** بالحدّ الأدنى نفسه، وإلا حُسبت كل الصناديق الصغيرة التي يُصدرها الكاشف إيجابيات
   كاذبة. فالدقة عند ثقة ٠٫٥ تنتقل من ٠٫١٩ إلى ٠٫٤٦ بهذا السطر وحده.
٢. **صورتان من العشرين صعبتان عمدًا**، ويسمّيهما `HARD_IMAGES.txt`. وقد اختيرتا بالقياس: يجد
   `yolos-tiny` **صفر٪** من الكائنات المُعلَّمة في إحداهما و**٣٣٪** في الأخرى عند ثقة ٠٫٥. والمهمة
   ٢٫٥ عن هاتين.

**التشغيل الأول يُنزّل** `hustvl/yolos-tiny` من Hugging Face، نحو ٢٦ ميغابايت، ويُخزَّن بعدها.
ويستغرق الاستدلال على الصور العشرين كلها نحو ثانية على معالج حاسوب محمول.

</div>

## Setup

**A licence note before the model loads.** The core sections use `hustvl/yolos-tiny` through
`transformers`: YOLO-family, **Apache-2.0**, already pinned in `requirements.lock`. The popular
`ultralytics` package is **AGPL-3.0**, which obliges you to publish the source of anything you serve
with it. It lives in `requirements-optional.in` and is mentioned once, in the stretch section. This
distinction has cost real companies real money; it is worth ten seconds of your attention.

<div dir="rtl" align="right">

## الإعداد

**ملاحظة ترخيص قبل تحميل النموذج.** تستخدم الأقسام الأساسية `hustvl/yolos-tiny` عبر `transformers`:
من عائلة YOLO، برخصة **Apache-2.0**، ومثبَّتة في `requirements.lock`. أما حزمة `ultralytics` الشائعة
فرخصتها **AGPL-3.0** التي تُلزمك بنشر شيفرة كل ما تُقدّمه بها خدمةً. وهي في `requirements-optional.in`
وتُذكر مرّة واحدة في القسم الإضافي. وقد كلّف هذا التمييز شركات حقيقية مالًا حقيقيًا، فيستحقّ عشر ثوانٍ
من انتباهك.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset_dir, describe_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report

ensure("transformers", "torchmetrics", "faster-coco-eval", "matplotlib")
seed_everything(42)

import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image, ImageDraw

PHOTOS = get_dataset_dir("sample_photos") / "photos"
BOXES_DIR = get_dataset_dir("detection_boxes")
PHOTO_FILES = sorted(PHOTOS.glob("*.jpg"))

GROUND_TRUTH = pd.read_csv(BOXES_DIR / "boxes.csv")
CLASSES = (BOXES_DIR / "classes.txt").read_text().strip().split("\n")
CLASS_INDEX = {name: i for i, name in enumerate(CLASSES)}
MIN_BOX_AREA = 32 * 32          # the ground truth's size floor; predictions must use the same one

print(describe_dataset("detection_boxes"))
print(f"\n{len(PHOTO_FILES)} photographs | {len(GROUND_TRUTH)} boxes over "
      f"{GROUND_TRUTH.image.nunique()} images | classes {CLASSES}")
print(GROUND_TRUTH.groupby("class").size().to_string())
print(versions(), "| device:", device())

## Section 1 — Warm-up: the raw output, before it is tidied  (≈25 min)

Everything here works. Run the detector on three photographs and print what it actually returns,
before any post-processing. Then draw the boxes.

Look carefully at the shapes. `logits` is `(1, 100, 92)` and `pred_boxes` is `(1, 100, 4)`: the
model emits exactly **100 predictions per image, always**, one per query slot, and most of them are
the "no object" class. That fixed 100 becomes a variable-length answer only after you threshold it.

And the four numbers per box are **not pixels**. They are `(cx, cy, w, h)`, each normalised to the
image size. The cell prints one raw box next to its converted pixel form so you can see the
difference before it bites you in task 2.1.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: الخرج الخام قبل ترتيبه (نحو ٢٥ دقيقة)

كل ما هنا يعمل. شغّل الكاشف على ثلاث صور واطبع ما يُرجعه فعلًا قبل أي معالجة لاحقة. ثم ارسم الصناديق.

وانظر إلى الأشكال بدقّة. فـ`logits` شكله `(1, 100, 92)` و`pred_boxes` شكله `(1, 100, 4)`: أي أن
النموذج يُصدر **مئة تنبّؤ لكل صورة، دائمًا**، واحدًا لكل خانة استعلام، ومعظمها من فئة «لا كائن».
وهذه المئة الثابتة لا تصير إجابةً متغيّرة الطول إلا بعد أن تضع لها عتبة.

والأعداد الأربعة لكل صندوق **ليست بكسلات**، بل `(cx, cy, w, h)` مُطبَّعة على حجم الصورة. وتطبع الخلية
صندوقًا خامًا إلى جانب صورته بالبكسل لترى الفرق قبل أن يعضّك في المهمة ٢٫١.

</div>

In [ ]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection

MODEL_NAME = "hustvl/yolos-tiny"          # Apache-2.0, YOLO family, 6.5M parameters
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
detector = AutoModelForObjectDetection.from_pretrained(MODEL_NAME).eval()
LABELS = detector.config.id2label

image = Image.open(PHOTO_FILES[0]).convert("RGB")
with torch.no_grad():
    raw = detector(**processor(images=image, return_tensors="pt"))

print(f"{PHOTO_FILES[0].name}: {image.width} x {image.height} pixels")
print("logits    ", tuple(raw.logits.shape), "  <- 100 query slots x 92 classes")
print("pred_boxes", tuple(raw.pred_boxes.shape), "  <- 100 boxes, normalised (cx, cy, w, h)")

best = raw.logits.softmax(-1)[0, :, :-1].max(-1)
top = int(best.values.argmax())
cx, cy, w, h = raw.pred_boxes[0, top].tolist()
print(f"\nthe most confident slot is #{top}: {LABELS[int(best.indices[top])]} at "
      f"{float(best.values[top]):.2f}")
print(f"  raw box   (cx, cy, w, h) = ({cx:.3f}, {cy:.3f}, {w:.3f}, {h:.3f})   <- fractions of the image")
print(f"  in pixels (x1, y1, x2, y2) = "
      f"({(cx - w / 2) * image.width:.0f}, {(cy - h / 2) * image.height:.0f}, "
      f"{(cx + w / 2) * image.width:.0f}, {(cy + h / 2) * image.height:.0f})")

In [ ]:
import matplotlib.pyplot as plt

COLOURS = {"person": "#ff4d4d", "car": "#4d94ff", "stop sign": "#ffd24d"}


def draw(image, boxes, labels, scores=None, width=3):
    """Return a copy of `image` with pixel boxes drawn on it."""
    canvas = image.copy()
    pen = ImageDraw.Draw(canvas)
    for i, (box, label) in enumerate(zip(boxes, labels)):
        colour = COLOURS.get(label, "#66ff99")
        pen.rectangle([float(v) for v in box], outline=colour, width=width)
        caption = label if scores is None else f"{label} {scores[i]:.2f}"
        pen.text((float(box[0]) + 3, float(box[1]) + 3), caption, fill=colour)
    return canvas


def detect(path, threshold=0.5):
    """Run the detector on one photograph and return a DataFrame of pixel boxes."""
    photo = Image.open(path).convert("RGB")
    with torch.no_grad():
        outputs = detector(**processor(images=photo, return_tensors="pt"))
    kept = processor.post_process_object_detection(
        outputs, threshold=threshold, target_sizes=[(photo.height, photo.width)])[0]
    rows = []
    for score, label, box in zip(kept["scores"], kept["labels"], kept["boxes"]):
        name = LABELS[int(label)]
        if name in CLASS_INDEX:
            x1, y1, x2, y2 = [float(v) for v in box]
            rows.append({"image": path.name, "cls": name, "score": float(score),
                         "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                         "area": (x2 - x1) * (y2 - y1)})
    return photo, pd.DataFrame(rows)


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, path in zip(axes, PHOTO_FILES[:3]):
    photo, found = detect(path, threshold=0.5)
    ax.imshow(draw(photo, found[["x1", "y1", "x2", "y2"]].values, found["cls"], found["score"].values))
    ax.set_title(f"{path.name}\n{len(found)} boxes at conf 0.5")
    ax.axis("off")
plt.tight_layout()
plt.show()

**The thing to notice.** Three photographs, three different numbers of boxes. Every model you have
trained returned an answer of fixed shape — ten class probabilities, one price, one churn
probability. This one returns a *list*, and its length is part of the prediction.

That single fact is why detection needs its own metrics. You cannot take the accuracy of a list
against another list; you have to decide which predicted box corresponds to which real object
first, and *that* is what IoU is for.

**Change one thing:** lower `threshold` from 0.5 to 0.05 in the cell above and re-run. The pictures
fill with boxes. Nothing about the model changed — you changed what you were willing to believe.

<div dir="rtl" align="right">

**ما ينبغي ملاحظته.** ثلاث صور بثلاثة أعداد مختلفة من الصناديق. فكل نموذج درّبته أرجع إجابة ثابتة
الشكل: عشرة احتمالات فئات، أو سعرًا واحدًا، أو احتمال تسرّب واحدًا. وهذا يُرجع **قائمة**، وطولها جزء من
التنبّؤ.

وهذه الحقيقة وحدها هي سبب حاجة الكشف إلى مقاييسه الخاصة. فلا تستطيع أخذ دقة قائمة مقابل قائمة، بل
عليك أولًا أن تقرّر أي صندوق متنبَّأ به يقابل أي كائن حقيقي، و**لهذا** وُجد التداخل IoU.

**غيّر شيئًا واحدًا:** اخفض `threshold` من ٠٫٥ إلى ٠٫٠٥ في الخلية أعلاه وأعِد التشغيل. فتمتلئ الصور
بالصناديق. ولم يتغيّر في النموذج شيء — بل تغيّر ما أنت مستعدّ لتصديقه.

</div>

## Section 2 — Core: five tasks  (≈60 min)

1. `iou(box_a, box_b)`, matching the lecture's three numbers.
2. A confidence sweep: boxes, precision and recall at ten thresholds.
3. **NMS off, then on**, on a photograph with 33 boxes on 4 cars.
4. mAP@0.5 and mAP@0.75 against the ground truth.
5. The two hard photographs, and what would fix them.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: خمس مهام (نحو ٦٠ دقيقة)

١. `iou(box_a, box_b)` مطابقًا أرقام المحاضرة الثلاثة.
٢. مسح الثقة: الصناديق والدقة والاستدعاء عند عشر عتبات.
٣. **إطفاء كبت غير الأقصى ثم تشغيله** على صورة فيها ٣٣ صندوقًا على ٤ سيارات.
٤. mAP@0.5 وmAP@0.75 مقابل المرجع.
٥. الصورتان الصعبتان، وما الذي يُصلحهما.

</div>

### Task 2.1 — IoU, and the three numbers from the slide

Intersection over union. The intersection is the overlapping rectangle; the union is both areas
minus the overlap, because adding the two areas counts the overlap twice.

The three cases from this morning, all with box A = `(2, 2, 6, 6)`:

| B | intersection | union | IoU |
|---|---|---|---|
| `(4, 4, 8, 8)` | 2 × 2 = 4 | 16 + 16 − 4 = 28 | **0.143** |
| `(3, 3, 7, 7)` | 3 × 3 = 9 | 23 | **0.39** |
| `(2, 3, 6, 7)` | 4 × 3 = 12 | 20 | **0.60** |

Write `iou(box_a, box_b)` for `(x1, y1, x2, y2)` in pixels, and check it against
`torchvision.ops.box_iou`. Two traps, and both are in the assert:

- **Non-overlapping boxes.** `x2 - x1` goes negative and a careless implementation returns a
  *positive* IoU from two negative widths multiplied together. Clamp at zero.
- **The format.** These are corners. `yolos-tiny` gives you normalised centres. Write
  `cxcywh_to_xyxy` now, once, and use it everywhere below.

<div dir="rtl" align="right">

### المهمة ٢٫١ — التداخل، والأرقام الثلاثة من الشريحة

التقاطع على الاتّحاد. فالتقاطع هو المستطيل المشترك، والاتّحاد مجموع المساحتين ناقص التداخل، لأن جمع
المساحتين يعدّ التداخل مرّتين.

والحالات الثلاث من هذا الصباح، وفيها الصندوق أ = `(2, 2, 6, 6)`:

| ب | التقاطع | الاتّحاد | IoU |
|---|---|---|---|
| `(4, 4, 8, 8)` | ٢×٢ = ٤ | ١٦ + ١٦ − ٤ = ٢٨ | **٠٫١٤٣** |
| `(3, 3, 7, 7)` | ٣×٣ = ٩ | ٢٣ | **٠٫٣٩** |
| `(2, 3, 6, 7)` | ٤×٣ = ١٢ | ٢٠ | **٠٫٦٠** |

اكتب `iou(box_a, box_b)` لصيغة `(x1, y1, x2, y2)` بالبكسل، وتحقّق منها بـ`torchvision.ops.box_iou`.
وثمّة فخّان وكلاهما في الفحص:

- **الصناديق غير المتداخلة.** يصير `x2 - x1` سالبًا، فتُرجع الصيغة المتساهلة تداخلًا **موجبًا** من
  ضرب عرضين سالبين. فاحصر عند الصفر.
- **الصيغة.** هذه أركان، و`yolos-tiny` يعطيك مراكز مُطبَّعة. فاكتب `cxcywh_to_xyxy` الآن مرّة واحدة
  واستخدمها في كل ما بعد.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Intersection: the largest of the two left edges, the smallest of the two right
#    edges, and the same for top and bottom. Clamp the width and height at zero.
# 2) Union = area_a + area_b - intersection. Return 0.0 when the union is 0.
# 3) The converter: x1 = cx - w/2 and so on, then multiply by the image width and
#    height — the model's numbers are fractions, not pixels.
# 4) Check against torchvision.ops.box_iou, which takes two (N, 4) tensors.
# Search: "intersection over union numpy clamp zero"
# https://pytorch.org/vision/stable/generated/torchvision.ops.box_iou.html
#
# ١) التقاطع: أكبر الحافّتين اليسريين، وأصغر الحافّتين اليمنيين، ومثلهما للأعلى
#    والأسفل. واحصر العرض والارتفاع عند الصفر.
# ٢) الاتّحاد = مساحة أ + مساحة ب − التقاطع. وأرجِع ٠٫٠ إذا كان الاتّحاد صفرًا.
# ٣) المحوِّل: `x1 = cx - w/2` وهكذا، ثم اضرب في عرض الصورة وارتفاعها — فأعداد
#    النموذج كسور لا بكسلات.
# ٤) تحقّق بـ`torchvision.ops.box_iou` التي تأخذ مصفوفتين بشكل (N, 4).
# ابحث عن: "intersection over union numpy clamp zero"
# https://pytorch.org/vision/stable/generated/torchvision.ops.box_iou.html
# ────────────────────────────────────────────────────────────────────

    # TODO: Intersection, clamped at zero; union; their ratio, guarding a zero union.
    # مهمة: التقاطع محصورًا عند الصفر، ثم الاتّحاد، ثم نسبتهما مع حماية من اتّحاد صفري.
    # TODO: Corners from the centre and size, then scale by the image dimensions.
    # مهمة: الأركان من المركز والحجم، ثم القياس بأبعاد الصورة.
SLIDE_CASES = [((2, 2, 6, 6), (4, 4, 8, 8), 0.143),
               ((2, 2, 6, 6), (3, 3, 7, 7), 0.39),
               ((2, 2, 6, 6), (2, 3, 6, 7), 0.60)]

### Task 2.2 — the confidence sweep

One detector, ten thresholds, three numbers each: how many boxes survive, what fraction of them
were right (**precision**), and what fraction of the real objects you found (**recall**).

The matching rule, and it is the whole of detection evaluation: take your predictions in order of
confidence; a prediction is a true positive if it lands on an unclaimed ground-truth box of the
same class with IoU ≥ 0.5; otherwise it is a false positive. Ground-truth boxes nothing claimed are
false negatives. **One prediction per ground-truth box** — the second one on the same object is a
false positive, which is why task 2.3 matters.

Two rules for this dataset, both from "About the data":

- Keep only the three classes in `classes.txt`. `yolos-tiny` knows 91 and will happily report a
  traffic light.
- Drop predictions smaller than `MIN_BOX_AREA`, because the ground truth dropped them too. That one
  line moves precision at confidence 0.5 from 0.19 to 0.46 — measure it both ways if you want to
  see it.

Plot all three curves against the threshold. The shape is the trade-off every detection product
argues about: precision rises, recall falls, and where you sit on that curve is a business decision
rather than a technical one.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — مسح الثقة

كاشف واحد وعشر عتبات وثلاثة أعداد لكل عتبة: كم صندوقًا نجا، وكم نسبة الصحيح منها (**الدقة**)، وكم
نسبة الكائنات الحقيقية التي وجدتها (**الاستدعاء**).

وقاعدة المطابقة هي تقييم الكشف كلّه: خُذ تنبّؤاتك بترتيب الثقة؛ فالتنبّؤ إيجابي صحيح إذا وقع على صندوق
مرجعي غير مأخوذ من الفئة نفسها بتداخل ≥ ٠٫٥، وإلا فهو إيجابي كاذب. والصناديق المرجعية التي لم يأخذها
شيء سلبيات كاذبة. **وتنبّؤ واحد لكل صندوق مرجعي** — والثاني على الكائن نفسه إيجابي كاذب، ولهذا تهمّ
المهمة ٢٫٣.

وقاعدتان لهذه البيانات، وكلتاهما من «عن البيانات»:

- أبقِ الفئات الثلاث في `classes.txt` فقط. فـ`yolos-tiny` يعرف ٩١ فئة وسيبلّغ عن إشارة مرور بطيب خاطر.
- وأسقِط التنبّؤات الأصغر من `MIN_BOX_AREA` لأن المرجع أسقطها كذلك. وهذا السطر وحده ينقل الدقة عند
  ثقة ٠٫٥ من ٠٫١٩ إلى ٠٫٤٦ — فقِسها بالوجهين إن أردت أن ترى ذلك.

ارسم المنحنيات الثلاثة مقابل العتبة. والشكل هو المفاضلة التي يتجادل حولها كل منتج كشف: الدقة تصعد
والاستدعاء يهبط، وموضعك على المنحنى قرار تجاري لا تقني.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Run the detector once at a very low threshold and keep every box in a DataFrame.
#    Re-running it ten times is ten times the wait for the same numbers.
# 2) Filter to the three classes and to area >= MIN_BOX_AREA once, up front.
# 3) The matcher: sort by score, greedily claim the best unclaimed ground-truth box
#    of the same class with IoU >= 0.5, count TP / FP, and FN is what is left.
# 4) Loop the thresholds, call the matcher, collect rows, plot three lines.
# Search: "object detection precision recall greedy matching iou threshold"
# https://pytorch.org/vision/stable/generated/torchvision.ops.nms.html
#
# ١) شغّل الكاشف مرّة عند عتبة منخفضة جدًا واحتفظ بكل صندوق في `DataFrame`.
#    فتشغيله عشر مرّات انتظارٌ عشري للأرقام نفسها.
# ٢) رشِّح إلى الفئات الثلاث وإلى مساحة ≥ `MIN_BOX_AREA` مرّة واحدة في البداية.
# ٣) المُطابِق: رتّب بالدرجة، وخُذ جشعًا أفضل صندوق مرجعي غير مأخوذ من الفئة نفسها
#    بتداخل ≥ ٠٫٥، وعُدّ الإيجابيات الصحيحة والكاذبة، والباقي سلبيات كاذبة.
# ٤) كرّر على العتبات ونادِ المُطابِق واجمع الصفوف وارسم ثلاثة خطوط.
# ابحث عن: "object detection precision recall greedy matching iou threshold"
# https://pytorch.org/vision/stable/generated/torchvision.ops.nms.html
# ────────────────────────────────────────────────────────────────────

import time
# TODO: at or above MIN_BOX_AREA, and put everything in one DataFrame.
# مهمة: ≥ `MIN_BOX_AREA`، وضع الكل في `DataFrame` واحد.
    # TODO: Claim ground-truth boxes greedily in confidence order and count TP/FP/FN.
    # مهمة: خُذ الصناديق المرجعية جشعًا بترتيب الثقة وعُدّ الإيجابيات والسلبيات.
    kept = []
    for _, group in pred.groupby(["image", "cls"]):
        # TODO: Run torchvision.ops.nms on this group's boxes and scores, keep the winners.
        # مهمة: شغّل `torchvision.ops.nms` على صناديق هذه المجموعة ودرجاتها، وأبقِ الفائزين.
    return pd.concat(kept).reset_index(drop=True) if kept else pred
# TODO: For ten thresholds from 0.1 to 0.9, apply NMS, match, and record boxes/precision/recall.
# مهمة: لعشر عتبات من ٠٫١ إلى ٠٫٩، طبّق الكبت وطابِق وسجّل الصناديق والدقة والاستدعاء.
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(sweep.threshold, sweep.precision, marker="o", label="precision")
ax.plot(sweep.threshold, sweep.recall, marker="o", label="recall")
twin = ax.twinx()
ax.set_xlabel("confidence threshold"); ax.set_ylabel("precision / recall")
twin.set_ylabel("boxes kept")
ax.legend(loc="center left"); twin.legend(loc="center right"); ax.grid(alpha=0.3)
plt.title("one detector, ten thresholds")
plt.tight_layout(); plt.show()

### Task 2.3 — turn NMS off

The detector has 100 query slots and no idea that eleven of them are looking at the same car. Left
alone, it reports every one of them. Non-max suppression is the fix, and it is four lines: take the
highest-scoring box, delete every box of the same class that overlaps it by more than a threshold,
repeat.

First reproduce the lecture's example exactly. Five boxes, one image:

| box | coordinates | score |
|---|---|---|
| A | `(10, 10, 50, 50)` | 0.90 |
| B | `(14, 14, 54, 54)` | 0.85 |
| C | `(16, 16, 56, 56)` | 0.80 |
| D | `(12, 12, 52, 52)` | 0.40 |
| E | `(200, 200, 240, 240)` | 0.30 |

A's IoU against the others is `0.68, 0.57, 0.82, 0.00`. At a threshold of 0.5, B, C and D all go
and **`[0, 4]` survive** — A and E, two boxes for two objects.

Then the real thing. Find the photograph with the most raw car boxes, draw it twice — every box
above confidence 0.1, then the same photo after NMS — and put the two side by side. On the
reference run that photograph carries **33 raw car boxes over 4 real cars**, and NMS at 0.5 leaves
17. Not 4. The remainder are boxes that overlap too little to suppress each other, which is what a
threshold buys you and what it does not.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — أطفئ كبت غير الأقصى

للكاشف مئة خانة استعلام ولا فكرة لديه أن إحدى عشرة منها تنظر إلى السيارة نفسها. وإن تُرك وشأنه بلّغ
عنها كلها. وكبت غير الأقصى هو العلاج، وهو أربعة أسطر: خُذ أعلى صندوق درجةً، واحذف كل صندوق من الفئة
نفسها يتداخل معه فوق عتبة، وكرّر.

أعِد أولًا إنتاج مثال المحاضرة تمامًا. خمسة صناديق في صورة واحدة:

| الصندوق | الإحداثيات | الدرجة |
|---|---|---|
| أ | `(10, 10, 50, 50)` | ٠٫٩٠ |
| ب | `(14, 14, 54, 54)` | ٠٫٨٥ |
| ج | `(16, 16, 56, 56)` | ٠٫٨٠ |
| د | `(12, 12, 52, 52)` | ٠٫٤٠ |
| هـ | `(200, 200, 240, 240)` | ٠٫٣٠ |

وتداخل أ مع البقية `٠٫٦٨` و`٠٫٥٧` و`٠٫٨٢` و`٠٫٠٠`. وعند عتبة ٠٫٥ تذهب ب وج ود ويبقى **`[0, 4]`** —
أي أ وهـ، صندوقان لكائنين.

ثم الشيء الحقيقي. جِد الصورة ذات أكثر صناديق سيارات خامًا، وارسمها مرّتين — كل صندوق فوق ثقة ٠٫١، ثم
الصورة نفسها بعد الكبت — وضع الاثنتين جنبًا إلى جنب. وفي التشغيل المرجعي تحمل تلك الصورة **٣٣ صندوق
سيارة خامًا على ٤ سيارات حقيقية**، ويُبقي الكبت عند ٠٫٥ سبعة عشر. لا أربعة. والباقي صناديق تتداخل
أقلّ من أن يكبت بعضها بعضًا، وهذا ما تشتريه العتبة وما لا تشتريه.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The five slide boxes go into a (5, 4) float tensor and the scores into a (5,).
#    torchvision.ops.nms(boxes, scores, 0.5) returns the surviving indices.
# 2) Print A's IoU against each of the others first — the four numbers explain the
#    result, and the result on its own explains nothing.
# 3) For the photograph: group your low-threshold predictions by image, count the car
#    rows, take the largest.
# 4) Draw before and after on one figure with two panels, and title each with its
#    box count and the number of real cars in the ground truth.
# Search: "torchvision ops nms indices threshold"
# https://pytorch.org/vision/stable/generated/torchvision.ops.nms.html
#
# ١) تدخل صناديق الشريحة الخمسة في مصفوفة (٥، ٤) عشرية والدرجات في (٥،).
#    وتُرجع `torchvision.ops.nms(boxes, scores, 0.5)` فهارس الناجين.
# ٢) اطبع تداخل أ مع كلٍّ من البقية أولًا — فالأعداد الأربعة تشرح النتيجة، والنتيجة
#    وحدها لا تشرح شيئًا.
# ٣) وللصورة: جمّع تنبّؤاتك منخفضة العتبة حسب الصورة، وعُدّ صفوف السيارات،
#    وخُذ الأكبر.
# ٤) ارسم قبل وبعد في شكل واحد بلوحتين، وعنوِن كلًّا منهما بعدد صناديقها وعدد
#    السيارات الحقيقية في المرجع.
# ابحث عن: "torchvision ops nms indices threshold"
# https://pytorch.org/vision/stable/generated/torchvision.ops.nms.html
# ────────────────────────────────────────────────────────────────────

SLIDE_BOXES = torch.tensor([[10, 10, 50, 50], [14, 14, 54, 54], [16, 16, 56, 56],
                            [12, 12, 52, 52], [200, 200, 240, 240]], dtype=torch.float)
SLIDE_SCORES = torch.tensor([0.90, 0.85, 0.80, 0.40, 0.30])
# TODO: Find the photo with the most raw car boxes, then count them before and after NMS.
# مهمة: جِد الصورة ذات أكثر صناديق السيارات خامًا، ثم عُدّها قبل الكبت وبعده.
photo = Image.open(PHOTOS / BUSIEST).convert("RGB")
fig, (before, after) = plt.subplots(1, 2, figsize=(16, 6))
before.set_title(f"NMS off — {len(busy_before)} car boxes"); before.axis("off")
after.set_title(f"NMS at 0.5 — {len(busy_after)} boxes, for {REAL_CARS} real cars")
after.axis("off")
plt.tight_layout(); plt.show()

### Task 2.4 — mAP@0.5, and then mAP@0.75

Precision and recall depend on the threshold you happened to pick. Average precision does not: it
is the area under the precision-recall curve, computed over every threshold at once, and mAP is
that averaged over the classes. It is the number detection papers report and the number a client
will ask you for.

Use `torchmetrics`:

```python
MeanAveragePrecision(iou_type="bbox", backend="faster_coco_eval", iou_thresholds=[0.5, 0.75])
```

**The `backend` argument is not optional.** Without a COCO-eval backend torchmetrics raises
`ModuleNotFoundError`, and the default backend is `pycocotools`, which needs a C toolchain to
install. `faster-coco-eval` is the pure-wheel one, and it is what `requirements.lock` pins.

Compute it at both IoU thresholds. **mAP@0.75 will be lower** — the same boxes, judged on whether
they are in *nearly the right place* rather than *roughly the right place*. On the reference run:
**0.612 at 0.5, 0.526 at 0.75.** If your number lands in the same neighbourhood, your pipeline is
right; the assert bands it rather than pinning it, because it is a check on your plumbing and not a
score to beat.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — mAP@0.5 ثم mAP@0.75

تعتمد الدقة والاستدعاء على العتبة التي اخترتها مصادفةً. أما متوسّط الدقة فلا: فهو المساحة تحت منحنى
الدقة-الاستدعاء محسوبةً على كل العتبات معًا، وmAP متوسّطه على الفئات. وهو الرقم الذي تعرضه أوراق الكشف
والرقم الذي سيسألك عنه العميل.

استخدم `torchmetrics`:

```python
MeanAveragePrecision(iou_type="bbox", backend="faster_coco_eval", iou_thresholds=[0.5, 0.75])
```

**ووسيط `backend` ليس اختياريًا.** فبلا خلفية لتقييم COCO تُطلق torchmetrics خطأ
`ModuleNotFoundError`، والخلفية الافتراضية `pycocotools` تحتاج أدوات ترجمة C لتثبيتها. أما
`faster-coco-eval` فهي ذات العجلة الخالصة، وهي المثبَّتة في `requirements.lock`.

احسبه عند العتبتين. **وسيكون mAP@0.75 أقلّ** — الصناديق نفسها لكنها تُحاكَم على أنها في المكان
**الصحيح تقريبًا** لا **الصحيح إجمالًا**. وفي التشغيل المرجعي: **٠٫٦١٢ عند ٠٫٥ و٠٫٥٢٦ عند ٠٫٧٥**.
فإن وقع رقمك في الجوار نفسه فخط أنابيبك سليم؛ والفحص يضعه في نطاق لا يثبّته، لأنه فحص لسباكتك لا
درجة تتفوّق عليها.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) torchmetrics wants a list of dicts per image: predictions need boxes, scores and
#    labels; targets need boxes and labels. Labels are integers, so map the class
#    names through CLASS_INDEX.
# 2) Boxes are (N, 4) float tensors in xyxy pixels. An image with nothing detected
#    still needs an entry — an empty (0, 4) tensor, not a missing one.
# 3) Feed it the NMS-ed predictions at a low confidence: average precision integrates
#    over thresholds itself, so throwing boxes away first only lowers the score.
# 4) Read map_50 and map_75 out of the result dict.
# 5) Print map_per_class too, to see which of the three carries the score.
# Search: "torchmetrics MeanAveragePrecision backend faster_coco_eval"
# https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision.html
#
# ١) تريد torchmetrics قائمة قواميس لكل صورة: التنبّؤات تحتاج صناديق ودرجات
#    وتسميات، والأهداف تحتاج صناديق وتسميات. والتسميات أعداد صحيحة، فحوّل أسماء
#    الفئات عبر `CLASS_INDEX`.
# ٢) والصناديق مصفوفات (N, 4) عشرية بصيغة xyxy بالبكسل. والصورة التي لم يُكشف فيها
#    شيء تحتاج مُدخلًا أيضًا — مصفوفة فارغة (0, 4) لا مُدخلًا مفقودًا.
# ٣) أطعمه التنبّؤات بعد الكبت عند ثقة منخفضة: فمتوسّط الدقة يكامل على العتبات بنفسه،
#    ورمي الصناديق أولًا لا يفعل إلا خفض الدرجة.
# ٤) اقرأ `map_50` و`map_75` من قاموس النتيجة الذي تُرجعه `compute()`.
# ٥) واطبع كذلك `map_per_class` لترى أي الفئات الثلاث تحمل الدرجة.
# ابحث عن: "torchmetrics MeanAveragePrecision backend faster_coco_eval"
# https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision.html
# ────────────────────────────────────────────────────────────────────

from torchmetrics.detection import MeanAveragePrecision
    # TODO: Build the per-image dicts of boxes/labels (and scores for predictions).
    # مهمة: ابنِ قواميس كل صورة من الصناديق والتسميات (والدرجات للتنبّؤات).
# TODO: Score the NMS-ed predictions against the ground truth at IoU 0.5 and 0.75.
# مهمة: قيّم التنبّؤات بعد الكبت مقابل المرجع عند تداخل ٠٫٥ و٠٫٧٥.

### Task 2.5 — the two photographs it fails on

`HARD_IMAGES.txt` names two of the twenty. They were chosen by running this detector over 120
candidate photographs and taking the two where it recovered the least ground truth — 0% and 33% at
confidence 0.5.

Display both with their predictions and their ground truth drawn together. Then, for each one,
write two things in `FAILURE_NOTES`: the failure mode — **occlusion**, **unusual angle**, **small
objects**, or **a class it never saw** — and what would actually fix it. "More data" is not an
answer; *what kind* of data, or what change to the input, or what different model.

This is the task the TA walks the room for. The metric is 0.61; these two photographs are what 0.61
looks like from the inside.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — الصورتان اللتان يفشل فيهما

يسمّي `HARD_IMAGES.txt` صورتين من العشرين. وقد اختيرتا بتشغيل هذا الكاشف على ١٢٠ صورة مرشَّحة وأخذ
الاثنتين اللتين استرجع فيهما أقلّ قدر من المرجع: صفر٪ و٣٣٪ عند ثقة ٠٫٥.

اعرضهما مع تنبّؤاتهما ومرجعهما مرسومَين معًا. ثم اكتب لكلٍّ منهما شيئين في `FAILURE_NOTES`: نمط الفشل
— **حجب** أو **زاوية غير معتادة** أو **كائنات صغيرة** أو **فئة لم يرَها قط** — وما الذي يُصلحه فعلًا.
و«بيانات أكثر» ليست جوابًا؛ بل **أي نوع** من البيانات، أو أي تغيير في الدخل، أو أي نموذج مختلف.

وهذه هي المهمة التي يمشي المساعد في القاعة لأجلها. فالمقياس ٠٫٦١، وهاتان الصورتان هما ما يبدو عليه
٠٫٦١ من الداخل.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) HARD_IMAGES.txt sits next to the photos folder; its first field on each line is
#    the filename.
# 2) Draw predictions and ground truth on the same picture — different widths make
#    them tellable apart. A miss is a truth box with nothing on it.
# 3) Report each image's recall at confidence 0.5 so the words have a number beside
#    them.
# 4) Write the two notes. Name the mode, then name the fix.
# Search: "object detection failure modes occlusion scale viewpoint"
# https://arxiv.org/abs/1506.02640
#
# ١) يقع `HARD_IMAGES.txt` بجوار مجلّد الصور، وأول حقل في كل سطر
#    اسم الملف.
# ٢) ارسم التنبّؤات والمرجع على الصورة نفسها — واختلاف السماكة يميّزهما. والإخفاق
#    صندوق مرجعي لا شيء عليه.
# ٣) اعرض استدعاء كل صورة عند ثقة ٠٫٥ ليكون بجانب الكلام
#    رقم.
# ٤) اكتب الملاحظتين. سمِّ النمط ثم سمِّ العلاج.
# ابحث عن: "object detection failure modes occlusion scale viewpoint"
# https://arxiv.org/abs/1506.02640
# ────────────────────────────────────────────────────────────────────

HARD_IMAGES = [line.split()[0] for line in
               (PHOTOS.parent / "HARD_IMAGES.txt").read_text().strip().split("\n")]
# TODO: print the recall the detector achieved on it.
# مهمة: الكاشف فيها.
    # TODO: For each hard image: the failure mode, and a fix that is not just "more data".
    # مهمة: لكل صورة صعبة: نمط الفشل، وعلاج ليس مجرّد «بيانات أكثر».

## Section 3 — Stretch: fine-tune the detector, and watch it get worse  (≈30 min)

**Licence first.** Everything above used `hustvl/yolos-tiny` under Apache-2.0. If you go looking for
detection code after today you will land on `ultralytics`, which is excellent and is **AGPL-3.0** —
serve a model with it and you owe the world your source. It is in `requirements-optional.in`, not in
the course environment, and this is the only section that mentions it.

**The experiment.** Fine-tune this detector on the 100 extra images in `detection_boxes/images/`,
then re-score it on the same twenty photographs. The backbone stays frozen — only the detection and
classification heads train, 166,752 parameters out of 6.5M — six epochs, about 30 seconds on a
laptop CPU.

**What the reference run measured:**

| | training loss | mAP@0.5 | mAP@0.75 |
|---|---|---|---|
| before | — | 0.612 | 0.526 |
| after 6 epochs, heads only | 0.879 → 0.670 | **0.588** | 0.480 |
| after 6 epochs, everything unfrozen at 5e-5 | 1.082 → 0.991 | **0.523** | 0.214 |

The loss went down and the metric went down with it. That is the section. A hundred images cannot
teach a COCO-pretrained detector anything it does not already know about people and cars; what they
can do is drag it away from the general solution towards this small sample — and unfreezing the
backbone does that faster and further. It is the same failure as Thursday's run D, met a day early.

Two things to do with that: report the numbers you actually get, and write in `FINETUNE_VERDICT`
what evidence would have justified shipping the fine-tuned weights instead. If the loss falling is
not enough, say what is.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: اضبط الكاشف وراقبه يسوء (نحو ٣٠ دقيقة)

**الترخيص أولًا.** كل ما سبق استخدم `hustvl/yolos-tiny` برخصة Apache-2.0. وإن بحثت عن شيفرة كشف بعد
اليوم فستقع على `ultralytics`، وهي ممتازة ورخصتها **AGPL-3.0** — فمن قدّم بها نموذجًا خدمةً لزمه أن
يمنح العالم شيفرته. وهي في `requirements-optional.in` لا في بيئة المقرّر، وهذا القسم الوحيد الذي
يذكرها.

**التجربة.** اضبط هذا الكاشف على المئة صورة الإضافية في `detection_boxes/images/`، ثم أعِد تقييمه
على الصور العشرين نفسها. ويبقى العمود الفقري مجمَّدًا — فلا يتدرّب إلا رأسا الكشف والتصنيف، أي
١٦٦٬٧٥٢ معاملًا من ٦٫٥ ملايين — ستّ حقب، نحو ثلاثين ثانية على معالج حاسوب محمول.

**وما قاسه التشغيل المرجعي:**

| | خسارة التدريب | mAP@0.5 | mAP@0.75 |
|---|---|---|---|
| قبل | — | ٠٫٦١٢ | ٠٫٥٢٦ |
| بعد ٦ حقب، الرؤوس فقط | ٠٫٨٧٩ ← ٠٫٦٧٠ | **٠٫٥٨٨** | ٠٫٤٨٠ |
| بعد ٦ حقب، بلا تجميد عند 5e-5 | ١٫٠٨٢ ← ٠٫٩٩١ | **٠٫٥٢٣** | ٠٫٢١٤ |

نزلت الخسارة ونزل المقياس معها. وهذا هو القسم. فمئة صورة لا تستطيع أن تُعلّم كاشفًا مُدرَّبًا على COCO
شيئًا لا يعرفه عن الناس والسيارات؛ وما تستطيعه أن تجرّه بعيدًا عن الحلّ العام نحو هذه العيّنة الصغيرة
— وإلغاء تجميد العمود الفقري يفعل ذلك أسرع وأبعد. وهو الفشل نفسه الذي في تشغيلة «د» يوم الخميس،
لقيته قبله بيوم.

وأمران تفعلهما بذلك: اعرض الأرقام التي حصلت عليها فعلًا، واكتب في `FINETUNE_VERDICT` أي دليل كان
سيبرّر نشر الأوزان المضبوطة بدلًا من الأصلية. فإن لم يكفِ نزول الخسارة فقل ما الذي يكفي.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The processor takes annotations= in COCO format: a dict with image_id and a list
#    of {bbox: [x, y, w, h], category_id, area, iscrowd}. Note bbox is xywh here.
# 2) category_id must be the model's own label id, not your 0/1/2 — invert
#    detector.config.id2label to get the mapping.
# 3) Freeze by setting requires_grad = False on every parameter whose name starts
#    with the backbone's prefix, and pass only the rest to AdamW.
# 4) Re-run your task 2.4 scoring afterwards, on the same twenty photographs.
# Search: "huggingface fine-tune object detection transformers labels"
# https://huggingface.co/docs/transformers/tasks/object_detection
#
# ١) يأخذ المعالج `annotations=` بصيغة COCO: قاموس فيه `image_id` وقائمة من
#    {bbox: [x, y, w, h]، category_id، area، iscrowd}. ولاحظ أن `bbox` هنا xywh.
# ٢) ويجب أن يكون `category_id` معرّف التسمية عند النموذج لا ٠/١/٢ عندك — فاعكس
#    `detector.config.id2label` للحصول على التحويل.
# ٣) جمّد بجعل `requires_grad = False` لكل معامل يبدأ اسمه ببادئة العمود الفقري،
#    ومرّر الباقي فقط إلى `AdamW`.
# ٤) أعِد بعدها تشغيل تقييم المهمة ٢٫٤ على الصور العشرين نفسها.
# ابحث عن: "huggingface fine-tune object detection transformers labels"
# https://huggingface.co/docs/transformers/tasks/object_detection
# ────────────────────────────────────────────────────────────────────

EXTRA_IMAGES = sorted((BOXES_DIR / "images").glob("*.jpg"))
    # TODO: boxes, suppress, and score with the same metric as task 2.4.
    # مهمة: واكبت، وقيّم بالمقياس نفسه من المهمة ٢٫٤.
# TODO: Fine-tune a fresh copy with the backbone frozen, six epochs, then re-score it.
# مهمة: اضبط نسخة جديدة بعمود فقري مجمَّد، ستّ حقب، ثم أعِد تقييمها.
    # TODO: What evidence would have justified shipping these weights? The loss fell.
    # مهمة: أي دليل كان سيبرّر نشر هذه الأوزان؟ فالخسارة نزلت.

## Save your artefacts

`detections.parquet` is every kept box with what it matched: image, class, score, coordinates, the
ground-truth row it claimed, and the IoU of that claim. `pr_curve.png` is the sweep figure.

The matched columns are what makes the file worth keeping — a parquet of predictions alone is a
log; a parquet of predictions with their matches is an evaluation you can re-open and argue with.

<div dir="rtl" align="right">

## احفظ آثارك

`detections.parquet` هو كل صندوق أُبقي مع ما طابقه: الصورة والفئة والدرجة والإحداثيات وصف المرجع
الذي أخذه وتداخل ذلك الأخذ. و`pr_curve.png` هو شكل المسح.

وأعمدة المطابقة هي ما يجعل الملف يستحقّ الحفظ — فملفّ تنبّؤات وحده سجلّ، وملفّ تنبّؤات مع مطابقاتها
تقييمٌ تستطيع فتحه ومجادلته.

</div>

In [ ]:
matched_rows = []
claimed = set()
for row in scored.sort_values("score", ascending=False).itertuples():
    candidates = TRUTH[(TRUTH.image == row.image) & (TRUTH["class"] == row.cls)]
    best_index, best_iou = None, 0.0
    for candidate in candidates.itertuples():
        if candidate.Index in claimed:
            continue
        overlap = iou((row.x1, row.y1, row.x2, row.y2),
                      (candidate.x1, candidate.y1, candidate.x2, candidate.y2))
        if overlap > best_iou:
            best_index, best_iou = candidate.Index, overlap
    hit = best_iou >= 0.5
    if hit:
        claimed.add(best_index)
    matched_rows.append({"image": row.image, "cls": row.cls, "score": row.score,
                         "x1": row.x1, "y1": row.y1, "x2": row.x2, "y2": row.y2,
                         "matched_gt_id": int(best_index) if hit else -1,
                         "iou": round(best_iou, 4)})

detections = pd.DataFrame(matched_rows)
DETECTIONS_PATH = ARTEFACT_DIR / "detections.parquet"
detections.to_parquet(DETECTIONS_PATH, index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sweep.recall, sweep.precision, marker="o")
for row in sweep.itertuples():
    ax.annotate(f"{row.threshold:.1f}", (row.recall, row.precision), fontsize=8,
                textcoords="offset points", xytext=(4, 4))
ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_title(f"precision against recall, labelled by confidence — mAP@0.5 = {MAP_50:.3f}")
ax.grid(alpha=0.3)
plt.tight_layout()
PR_CURVE_PATH = ARTEFACT_DIR / "pr_curve.png"
plt.savefig(PR_CURVE_PATH, dpi=110)
plt.show()

print(f"wrote {DETECTIONS_PATH.name} ({len(detections)} rows, "
      f"{int((detections.matched_gt_id >= 0).sum())} matched) and {PR_CURVE_PATH.name}")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(all(np.isclose(iou(a, b), expected, atol=5e-3) for a, b, expected in SLIDE_CASES),
      f"iou() must reproduce the lecture's 0.143, 0.39 and 0.60 to the slide's precision — got "
      f"{[round(iou(a, b), 3) for a, b, _ in SLIDE_CASES]}",
      f"يجب أن تُعيد `iou()` أرقام المحاضرة ٠٫١٤٣ و٠٫٣٩ و٠٫٦٠، والناتج "
      f"{[round(iou(a, b), 3) for a, b, _ in SLIDE_CASES]}")

check(SURVIVORS == [0, 4],
      f"NMS on the lecture's five boxes must leave [0, 4] — got {SURVIVORS}",
      f"يجب أن يُبقي الكبت على صناديق المحاضرة الخمسة `[0, 4]`، والناتج {SURVIVORS}")

check(BIG_ENOUGH.score.between(0, 1).all(),
      f"every confidence score must lie in [0, 1] — range is "
      f"{BIG_ENOUGH.score.min():.3f} to {BIG_ENOUGH.score.max():.3f}",
      f"يجب أن تقع كل درجة ثقة في [٠، ١]، والمدى {BIG_ENOUGH.score.min():.3f} إلى "
      f"{BIG_ENOUGH.score.max():.3f}")

check(sweep.boxes.is_monotonic_decreasing,
      f"raising the threshold can only remove boxes — the counts were {sweep.boxes.tolist()}",
      f"رفع العتبة لا يفعل إلا حذف الصناديق — والأعداد كانت {sweep.boxes.tolist()}")

check(len(busy_after) < len(busy_before),
      f"NMS must leave strictly fewer boxes on the busy photo — {len(busy_before)} before, "
      f"{len(busy_after)} after",
      f"يجب أن يُبقي الكبت صناديق أقلّ قطعًا في الصورة المزدحمة — {len(busy_before)} قبل و"
      f"{len(busy_after)} بعد")

check(0.45 <= MAP_50 <= 0.80 and MAP_75 < MAP_50,
      f"mAP@0.5 should land in the sane band 0.45-0.80 for yolos-tiny on this set (a check on "
      f"your pipeline, not a target to beat) and mAP@0.75 must be lower — got "
      f"{MAP_50:.3f} and {MAP_75:.3f}",
      f"يجب أن يقع mAP@0.5 في النطاق المعقول ٠٫٤٥–٠٫٨٠ لـyolos-tiny على هذه المجموعة (وهو فحص "
      f"لخط أنابيبك لا هدف تتفوّق عليه) وأن يكون mAP@0.75 أقلّ — والناتج "
      f"{MAP_50:.3f} و{MAP_75:.3f}")

check(set(FAILURE_NOTES) == set(HARD_IMAGES)
      and all(len(note.split()) >= 15 for note in FAILURE_NOTES.values()),
      f"both hard images need a written finding of at least 15 words — got "
      f"{ {name: len(note.split()) for name, note in FAILURE_NOTES.items()} }",
      f"تحتاج الصورتان الصعبتان كلتاهما نتيجةً مكتوبة لا تقلّ عن ١٥ كلمة — والموجود "
      f"{ {name: len(note.split()) for name, note in FAILURE_NOTES.items()} }")

report()

## What's next

**W4D4 — Augmentation A/B.** Tomorrow the dataset is 400 images across five classes, which is too
few, and the question is what augmentation actually buys you — measured, with the same seed and the
same architecture on both sides, rather than assumed.

You met the argument for it today without being told: the second hard photograph fails on a
viewpoint the training data barely contained, and manufacturing missing viewpoints from the data
you already have is exactly what augmentation is.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٤ اليوم ٤ — زيادة البيانات: تجربة أ/ب.** غدًا تكون البيانات أربعمئة صورة في خمس فئات، وهي
أقلّ من اللازم، والسؤال ما الذي تشتريه زيادة البيانات فعلًا — مقيسًا بالبذرة نفسها والبنية نفسها في
الجانبين، لا مفترَضًا.

وقد لقيت حجّتها اليوم دون أن يُقال لك: فالصورة الصعبة الثانية تفشل بزاوية لم تكن بيانات التدريب تحوي
منها إلا القليل، وتصنيعُ الزوايا المفقودة من بيانات تملكها هو بالضبط ما تكونه زيادة البيانات.

</div>